# pipe_nuevo — 05 DTW clusters: comparar varios k de una corrida

Va DESPUES de `03_Escalado` (y de la revision de `04_Explorar_features`) en el
orden de pipe_nuevo. Arma las series por PAR cliente-producto igual que
`nat_exp/dtw_nuevo.ipynb` (misma densificacion, mismo escalado propio de la
serie completa, mismo `k-means` con DTW y centroides DBA para no depender de
la matriz n^2 que hace inviable el nivel de par en `02_DTW_clusters.ipynb`) --
pero lee el CACHE de `02_FE` (`features_sin_escalar_*.parquet`) en vez del
parquet del pipe viejo, para no depender de haber corrido nada fuera de
pipe_nuevo.

En vez de ajustar UN k fijo, corre `PARAM['lista_k']` completa sobre las
MISMAS series ya armadas -- el trabajo caro (densificar + escalar) se paga una
sola vez, y lo que se repite por cada k es solo el ajuste de k-means (que
igual es el costo dominante).

Por cada k deja: tabla de composicion + lift por cat1/cat2/cat3/brand (con el
top-3 sobrerrepresentado por cluster), heatmap de lift por cat3, y la forma de
cada cluster (percentil 10-90 + centroide DBA). Al final arma una tabla
comparativa (silhouette, balance, concentracion por cat3 dominante) para
decidir que k tiene sentido, y deja las etiquetas del k elegido guardadas en
`datasets_fe/` listas para unir mas adelante (por `product_id`+`customer_id`).


In [ ]:
import gc, json, os, time
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from dtaidistance import dtw
from sklearn.metrics import silhouette_score


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local ultimo."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    local = Path(r"C:\Users\Natalia\labo3-bucket")
    if local.is_dir():
        return local
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


BUCKET   = resolver_bucket()
DIR_RAW  = BUCKET / "datasets"
RUTA_FE  = BUCKET / "datasets_fe"        # de aca lee (cache de 02_FE) y aca escribe
DIR_RUNS = BUCKET / "exp_clusters_pc"
RUTA_FE.mkdir(parents=True, exist_ok=True)
DIR_RUNS.mkdir(parents=True, exist_ok=True)

SERIE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
         "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
TINTA, TINTA2, MUDO = "#0b0b0b", "#52514e", "#898781"
GRILLA, EJE_C, FONDO = "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO,
    "axes.edgecolor": EJE_C, "axes.labelcolor": TINTA2,
    "text.color": TINTA, "xtick.color": MUDO, "ytick.color": MUDO,
    "grid.color": GRILLA, "grid.linewidth": .8,
    "axes.grid": True, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "figure.dpi": 110,
    "legend.frameon": False,
})


def limpiar(ax, titulo=None, y=None, x=None):
    if titulo:
        ax.set_title(titulo, color=TINTA, loc="left", pad=10)
    if y:
        ax.set_ylabel(y)
    if x:
        ax.set_xlabel(x)
    ax.grid(axis="x", visible=False)
    return ax


def guardar(fig, nombre, subcarpeta=None, mostrar=True):
    d = DIR_RUN if subcarpeta is None else DIR_RUN / subcarpeta
    d.mkdir(parents=True, exist_ok=True)
    p = d / f"{nombre}.png"
    fig.savefig(p, dpi=140, bbox_inches="tight", facecolor=FONDO)
    print(f"   [fig] {p.relative_to(BUCKET)}")
    plt.show() if mostrar else plt.close(fig)
    return p


_C_OK = dtw.try_import_c()
print(f"BUCKET: {BUCKET}")
print(f"DTW con backend C: {_C_OK}")
if not _C_OK:
    print("   ATENCION: sin backend C esto es inusable para 370k series. pip install -U dtaidistance")

try:
    dtw.warping_path(np.zeros(5), np.zeros(5), window=2)
    _WP_WINDOW = True
except TypeError:
    _WP_WINDOW = False
print(f"warping_path acepta window: {_WP_WINDOW}")


### Palancas

Las mismas de `dtw_nuevo.ipynb` para armar las series (fuente, densificacion, corte anti-leakage, escalado, window), mas `lista_k` en vez de un `k` fijo -- eso es lo unico nuevo.


In [ ]:
PARAM = {
    # ── DE DONDE SALEN LAS SERIES (igual que dtw_nuevo) ──────────────────
    'fuente': 'preprocesado',
    'archivo_preprocesado': None,
    'densificar': 'desde_nacimiento',
    'mes_corte': 201906,
    'min_meses': 18,
    'escalado': 'media',
    'window': 3,

    # ── K-MEANS CON DTW ───────────────────────────────────────────────────
    'lista_k': [3, 4, 5, 6, 8],
    'max_iter': 15,
    'tol_cambio': 0.01,
    'dba_iters': 1,

    # ── MUESTREO ──────────────────────────────────────────────────────────
    'muestra_pares': None,
    'muestra_silhouette': 3000,

    # ── EXPLORACION POR K ────────────────────────────────────────────────
    # Grafica forma-por-cluster y heatmap de lift para cada k de la lista. Con
    # listas largas (>6 valores de k) conviene apagarlo y mirar solo la tabla
    # comparativa primero.
    'graficar_por_k': True,
    'min_pares_categoria': 20,   # una categoria entra al top-3 sobrerrepresentado
                                 # de un cluster solo si tiene al menos esto

    'semilla': 102191,
}

ESCALADOS = ('media', 'zscore', 'maximo', 'ninguno')
if PARAM['escalado'] not in ESCALADOS:
    raise ValueError(f"escalado invalido: {PARAM['escalado']!r}. Opciones: {ESCALADOS}")
if PARAM['densificar'] not in ('desde_nacimiento', 'vida'):
    raise ValueError(f"densificar invalido: {PARAM['densificar']!r}")
if PARAM['fuente'] not in ('crudo', 'preprocesado'):
    raise ValueError(f"fuente invalida: {PARAM['fuente']!r}")

RNG = np.random.default_rng(PARAM['semilla'])
CATS = ["cat1", "cat2", "cat3", "brand"]

SLUG = (f"pc_{PARAM['densificar']}_{PARAM['escalado']}"
        f"_w{PARAM['window']}_min{PARAM['min_meses']}_corte{PARAM['mes_corte']}")
DIR_RUN = DIR_RUNS / SLUG / "explorar_k"
DIR_RUN.mkdir(parents=True, exist_ok=True)
with open(DIR_RUN / "config.json", "w", encoding="utf-8") as f:
    json.dump(PARAM, f, indent=2, ensure_ascii=False)

print(f"carpeta : {DIR_RUN.relative_to(BUCKET)}")
print(f"lista_k : {PARAM['lista_k']}")
print(f"escalado: {PARAM['escalado']}   window: {PARAM['window']}")


### Armado de las series (identico a `dtw_nuevo.ipynb`)


In [ ]:
t0 = time.time()


def a_m(p):
    """AAAAMM -> indice de mes continuo, para poder sumar y restar meses."""
    return (p // 100) * 12 + (p % 100)


def m_a_periodo(m):
    return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1


def normalizar_periodo(df: pl.DataFrame) -> pl.DataFrame:
    """Deja 'periodo' como Int64 AAAAMM, venga como Date, string o entero."""
    dt = df.schema['periodo']
    try:
        temporal = dt.is_temporal()
    except AttributeError:
        temporal = dt in (pl.Date, pl.Datetime)
    if temporal:
        print(f"  periodo venia como {dt} -> se convierte a Int64 AAAAMM")
        return df.with_columns(
            (pl.col('periodo').dt.year() * 100 + pl.col('periodo').dt.month())
            .cast(pl.Int64).alias('periodo'))
    if dt == pl.Utf8:
        print("  periodo venia como texto -> se convierte a Int64 AAAAMM")
        return df.with_columns(
            pl.col('periodo').str.replace_all(r'\D', '').str.slice(0, 6)
              .cast(pl.Int64).alias('periodo'))
    return df.with_columns(pl.col('periodo').cast(pl.Int64))


if PARAM['fuente'] == 'preprocesado':
    # Cache de 02_FE: ya viene con las categorias pegadas y densificado con el
    # zero-fill cartesiano de 01_Preprocesamiento (cliente x producto x periodo,
    # no solo la vida propia del par -- por eso 'vida' abajo se calcula SOLO con
    # ventas reales, no con todas las filas).
    disp = sorted(RUTA_FE.glob("features_sin_escalar_*.parquet"))
    if not disp:
        raise FileNotFoundError(f"No hay features_sin_escalar_*.parquet en {RUTA_FE}. "
                                f"Corre 01_Preprocesamiento -> 02_FE primero, o usa fuente='crudo'.")
    if PARAM['archivo_preprocesado']:
        path_pre = RUTA_FE / PARAM['archivo_preprocesado']
        if not path_pre.exists():
            raise FileNotFoundError(f"No existe {path_pre}.\nDisponibles: "
                                    f"{[p.name for p in disp]}")
    else:
        path_pre = max(disp, key=lambda p: p.stat().st_mtime)
    print(f"Leyendo {path_pre.name}")

    raw = pl.read_parquet(path_pre)
    if "tn0" in raw.columns and "tn" not in raw.columns:
        raw = raw.rename({"tn0": "tn"})
    faltan = [c for c in ["product_id", "customer_id", "periodo", "tn"] if c not in raw.columns]
    if faltan:
        raise ValueError(f"El parquet no tiene {faltan}. Columnas: {raw.columns}")
    raw = normalizar_periodo(raw)
    cats_ok = [c for c in CATS if c in raw.columns]
    panel = (raw.group_by(["product_id", "customer_id", "periodo"])
                .agg(pl.col("tn").sum().alias("tn"))
                .join(raw.select(["product_id"] + cats_ok).unique(subset=["product_id"]),
                      on="product_id", how="left"))
else:
    sell = normalizar_periodo(pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t"))
    prod = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
              .unique(subset=["product_id"]))
    cats_ok = [c for c in CATS if c in prod.columns]
    panel = (sell.group_by(["product_id", "customer_id", "periodo"])
                 .agg(pl.col("tn").sum().alias("tn"))
                 .join(prod.select(["product_id"] + cats_ok), on="product_id", how="left"))

panel = panel.with_columns(a_m(pl.col("periodo")).alias("m"))

if PARAM['mes_corte'] is not None:
    m_corte = a_m(PARAM['mes_corte'])
    antes = panel.height
    panel = panel.filter(pl.col("m") < m_corte)
    print(f"Corte anti-leakage en {PARAM['mes_corte']}: {antes:,} -> {panel.height:,} filas")

M_MIN, M_MAX = int(panel["m"].min()), int(panel["m"].max())
print(f"panel: {panel.height:,} filas · "
      f"{panel['product_id'].n_unique()} productos x {panel['customer_id'].n_unique()} clientes")
print(f"meses {m_a_periodo(M_MIN)} -> {m_a_periodo(M_MAX)}  ({M_MAX - M_MIN + 1} meses)")
print(f"[{time.time()-t0:.0f}s]")


In [ ]:
t0 = time.time()
KEYS = ["product_id", "customer_id"]

# OJO: m_nace/m_ultima se calculan SOLO con filas de venta real (tn > 0). El
# panel de 'preprocesado' viene del cache de 02_FE, que ya esta densificado con
# CEROS EXPLICITOS (cliente x producto x periodo, no solo la vida del par) --
# si se calculara sobre todas las filas, m_nace quedaria en el momento en que
# cliente y producto EMPEZARON A COEXISTIR, no en el que el par empezo a
# comprar de verdad, y la serie 'nacería' antes de lo que nació.
vida = (panel.filter(pl.col("tn") > 0)
             .group_by(KEYS)
             .agg(pl.col("m").min().alias("m_nace"),
                  pl.col("m").max().alias("m_ultima"),
                  pl.col("tn").sum().alias("tn_total"),
                  pl.len().alias("meses_con_venta")))

if PARAM['densificar'] == 'desde_nacimiento':
    vida = vida.with_columns(pl.lit(M_MAX).alias("m_fin"))
else:
    vida = vida.with_columns(pl.col("m_ultima").alias("m_fin"))
vida = vida.with_columns((pl.col("m_fin") - pl.col("m_nace") + 1).alias("largo"))

elegibles = vida.filter(pl.col("largo") >= PARAM['min_meses'])
print(f"pares: {vida.height:,} totales -> {elegibles.height:,} con >= "
      f"{PARAM['min_meses']} meses de serie")
if elegibles.height == 0:
    raise ValueError(
        f"Ningun par tiene >= {PARAM['min_meses']} meses de serie (el mas largo "
        f"tiene {int(vida['largo'].max()) if vida.height else 0}). Baja "
        f"PARAM['min_meses'] o revisa PARAM['mes_corte']."
    )

if PARAM['muestra_pares'] and elegibles.height > PARAM['muestra_pares']:
    elegibles = elegibles.sort("tn_total", descending=True).head(PARAM['muestra_pares'])
    print(f"  muestreados a los {PARAM['muestra_pares']:,} de mayor tn")

grilla = (elegibles.select(KEYS + ["m_nace", "m_fin"])
                   .with_columns(pl.int_ranges("m_nace", pl.col("m_fin") + 1).alias("m"))
                   .explode("m")
                   .select(KEYS + ["m"]))
denso = (grilla.join(panel.select(KEYS + ["m", "tn"]), on=KEYS + ["m"], how="left")
               .with_columns(pl.col("tn").fill_null(0.0))
               .sort(KEYS + ["m"]))

_ceros = int((denso["tn"] == 0).sum())
print(f"panel denso: {denso.height:,} filas ({_ceros:,} ceros = {100*_ceros/denso.height:.0f}%)")
print(f"[{time.time()-t0:.0f}s]")


def escalar(v: np.ndarray, modo: str) -> np.ndarray:
    v = np.asarray(v, dtype=np.float64)
    if modo == 'media':
        mu = v.mean()
        out = v / mu if abs(mu) > 1e-9 else v
    elif modo == 'zscore':
        sd = v.std()
        out = (v - v.mean()) / sd if sd > 1e-9 else v - v.mean()
    elif modo == 'maximo':
        mx = np.abs(v).max()
        out = v / mx if mx > 1e-9 else v
    elif modo == 'ninguno':
        out = v
    else:
        raise ValueError(f"escalado invalido: {modo!r}")
    return np.ascontiguousarray(out, dtype=np.float64)


def armar_series(denso_df: pl.DataFrame, modo: str):
    g = (denso_df.sort(KEYS + ["m"])
                 .group_by(KEYS, maintain_order=True)
                 .agg(pl.col("tn").alias("serie"), pl.col("m").min().alias("m0")))
    pares = list(zip(g["product_id"].to_list(), g["customer_id"].to_list()))
    crudas = [np.asarray(s, dtype=np.float64) for s in g["serie"].to_list()]
    escaladas = [escalar(s, modo) for s in crudas]
    return pares, escaladas, crudas, g["m0"].to_list()


PARES, SERIES, CRUDAS, M0 = armar_series(denso, PARAM['escalado'])
LARGOS = np.array([len(s) for s in SERIES])
print(f"\n{len(SERIES):,} series   escalado: {PARAM['escalado']}")
print(f"largo  min/mediana/max: {LARGOS.min()} / {int(np.median(LARGOS))} / {LARGOS.max()}")


### Motor DTW (identico a `dtw_nuevo.ipynb`): k-means con centroides DBA, memoria O(n x k)


In [ ]:
def banda(a, b, window):
    """Banda de Sakoe-Chiba factible: se ensancha lo justo si los largos difieren
    mas que 'window', para no devolver inf en pares que nacieron en meses distintos."""
    if window is None:
        return None
    return max(int(window), abs(len(a) - len(b)))


def d_dtw(a, b, window=None):
    return dtw.distance_fast(a, b, window=banda(a, b, window), use_pruning=False)


def sanear(D):
    mal = ~np.isfinite(D)
    if not mal.any():
        return D, 0
    fin = D[~mal]
    D[mal] = (fin.max() * 10.0) if fin.size else 1.0
    return D, int(mal.sum())


def asignar(series, centros, window=None):
    n, k = len(series), len(centros)
    D = np.empty((n, k), dtype=np.float64)
    for j, c in enumerate(centros):
        cj = np.ascontiguousarray(c, dtype=np.float64)
        for i, s in enumerate(series):
            D[i, j] = d_dtw(s, cj, window)
    D, n_mal = sanear(D)
    if n_mal:
        print(f"   aviso: {n_mal:,} de {n*k:,} distancias no finitas (banda infactible)")
    lab = D.argmin(axis=1)
    return lab, D[np.arange(n), lab], D


def dba(miembros, centro, window=None, iters=1):
    centro = np.ascontiguousarray(centro, dtype=np.float64)
    if not miembros:
        return centro
    T = len(centro)
    for _ in range(iters):
        acum = np.zeros(T, dtype=np.float64)
        cuenta = np.zeros(T, dtype=np.float64)
        for s in miembros:
            w = banda(centro, s, window)
            path = (dtw.warping_path(centro, s, window=w) if _WP_WINDOW
                    else dtw.warping_path(centro, s))
            for i, j in path:
                acum[i] += s[j]
                cuenta[i] += 1.0
        centro = np.where(cuenta > 0, acum / np.maximum(cuenta, 1.0), centro)
        centro = np.ascontiguousarray(centro, dtype=np.float64)
    return centro


def init_kmeanspp(series, k, window, rng):
    n = len(series)
    centros = [series[int(rng.integers(n))].copy()]
    d_min = np.array([d_dtw(s, centros[0], window) for s in series])
    for _ in range(1, k):
        p = d_min ** 2
        tot = p.sum()
        idx = int(rng.integers(n)) if tot <= 0 else int(rng.choice(n, p=p / tot))
        centros.append(series[idx].copy())
        d_nuevo = np.array([d_dtw(s, centros[-1], window) for s in series])
        d_min = np.minimum(d_min, d_nuevo)
    return centros


def kmeans_dtw(series, k, window=None, max_iter=15, tol=0.01, dba_iters=1,
               semilla=0, verbose=True):
    rng = np.random.default_rng(semilla)
    n = len(series)
    centros = init_kmeanspp(series, k, window, rng)
    lab = np.full(n, -1)
    hist = []
    for it in range(1, max_iter + 1):
        lab_new, d_prop, _ = asignar(series, centros, window)
        cambios = int((lab_new != lab).sum())
        lab = lab_new
        inercia = float(d_prop.sum())
        hist.append({'iter': it, 'inercia': inercia, 'cambios': cambios,
                     'frac_cambios': cambios / n})
        if verbose:
            tam = np.bincount(lab, minlength=k)
            print(f"  iter {it:2d}  inercia {inercia:12,.1f}  "
                  f"cambian {cambios:6,} ({100*cambios/n:5.2f}%)  tam {tam.tolist()}")
        for j in range(k):
            if not np.any(lab == j):
                peor = int(np.argmax(d_prop))
                centros[j] = series[peor].copy()
                lab[peor] = j
        centros = [dba([series[i] for i in np.flatnonzero(lab == j)], centros[j],
                       window, dba_iters)
                   for j in range(k)]
        if cambios / n <= tol and it > 1:
            if verbose:
                print(f"  convergio: cambia {100*cambios/n:.2f}% <= {100*tol:.2f}%")
            break
    lab, d_prop, D = asignar(series, centros, window)
    return {'labels': lab, 'centros': centros, 'inercia': float(d_prop.sum()),
            'dist_propio': d_prop, 'D': D, 'hist': hist, 'k': k, 'window': window}


def matriz_dtw(series, window=None):
    """Matriz simetrica completa. Solo para muestras chicas: es O(n^2)."""
    n = len(series)
    D = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        si = series[i]
        for j in range(i + 1, n):
            D[i, j] = D[j, i] = d_dtw(si, series[j], window)
    D, _ = sanear(D)
    return D


def silhouette_muestra(series, labels, window, n_muestra, rng):
    labels = np.asarray(labels)
    n = len(series)
    if n <= n_muestra:
        idx = np.arange(n)
    else:
        idx = []
        for j in np.unique(labels):
            en_j = np.flatnonzero(labels == j)
            cuota = max(2, int(round(n_muestra * len(en_j) / n)))
            cuota = min(cuota, len(en_j))
            idx.append(rng.choice(en_j, size=cuota, replace=False))
        idx = np.concatenate(idx)
    lab_m = labels[idx]
    if len(np.unique(lab_m)) < 2:
        return float('nan'), len(idx)
    D = matriz_dtw([series[i] for i in idx], window)
    return float(silhouette_score(D, lab_m, metric='precomputed')), len(idx)


def resumen_particion(labels, dist_propio, series):
    labels = np.asarray(labels)
    tam = np.bincount(labels, minlength=labels.max() + 1)
    return {
        'k_efectivo': int((tam > 0).sum()),
        'tam_min': int(tam.min()), 'tam_max': int(tam.max()),
        'frac_min': float(tam.min() / len(labels)),
        'inercia': float(np.sum(dist_propio)),
        'inercia_media': float(np.mean(dist_propio)),
    }


print("motor DTW listo")


### Composicion por categoria + lift (NUEVO: empaquetado como funcion para poder correrlo por cada k)

`lift > 1` = esa categoria esta sobrerrepresentada en el cluster respecto de su peso global; `< 1` sub-representada. `concentracion_cat3`: cuanto del cluster se lo lleva su cat3 dominante -- alto (>0.5) significa que el cluster es casi una sola categoria y aporta poco sobre lo que ya sabiamos por la jerarquia comercial; repartido = agrupa por FORMA cruzando categorias, que es informacion nueva.


In [ ]:
cats_disp = [c for c in CATS if c in panel.columns]
pc_cat = panel.select(KEYS + cats_disp).unique(subset=KEYS)


def explorar_k(k, dir_k, graficar=True):
    """Ajusta k-means DTW para este k sobre las SERIES ya armadas, arma la
    composicion por categoria + lift, y opcionalmente grafica forma-por-cluster
    y el heatmap de lift por cat3. Devuelve (fila_comparacion, resumen_cl, etiquetas).
    """
    t0 = time.time()
    print(f"\n{'='*74}\nk = {k}\n{'='*74}")
    RES = kmeans_dtw(SERIES, k, window=PARAM['window'], max_iter=PARAM['max_iter'],
                     tol=PARAM['tol_cambio'], dba_iters=PARAM['dba_iters'],
                     semilla=PARAM['semilla'], verbose=False)
    LAB = RES['labels']
    sil, n_sil = silhouette_muestra(SERIES, LAB, PARAM['window'],
                                    PARAM['muestra_silhouette'],
                                    np.random.default_rng(PARAM['semilla']))
    resumen = resumen_particion(LAB, RES['dist_propio'], SERIES)
    print(f"silhouette (sobre {n_sil:,} series): {sil:+.4f}   {resumen}")

    tn_par = np.array([c.sum() for c in CRUDAS])
    meses_con_venta = np.array([int((c > 0).sum()) for c in CRUDAS])
    etiquetas = pl.DataFrame({
        'product_id':  [p for p, _ in PARES],
        'customer_id': [c for _, c in PARES],
        'cluster':     LAB.astype(np.int32),
        'dist_centroide': RES['dist_propio'],
        'tn_total':    tn_par,
        'largo':       LARGOS,
        'meses_con_venta': meses_con_venta,
    }).with_columns((pl.col("meses_con_venta") / pl.col("largo")).alias("frac_meses_con_venta"))

    resumen_cl = (etiquetas.group_by("cluster")
        .agg(pl.len().alias("n_pares"),
             pl.col("product_id").n_unique().alias("n_productos"),
             pl.col("customer_id").n_unique().alias("n_clientes"),
             pl.col("tn_total").sum().alias("tn"),
             pl.col("tn_total").median().alias("tn_mediana_par"),
             pl.col("largo").median().alias("largo_mediano"),
             pl.col("frac_meses_con_venta").mean().alias("frac_meses_con_venta"),
             pl.col("dist_centroide").mean().alias("dist_media"))
        .with_columns((100 * pl.col("n_pares") / etiquetas.height).round(1).alias("pct_pares"),
                      (100 * pl.col("tn") / etiquetas["tn_total"].sum()).round(1).alias("pct_tn"))
        .sort("n_pares", descending=True))
    print(resumen_cl)

    et_cat = etiquetas.join(pc_cat, on=KEYS, how="left")
    orden_cl = resumen_cl["cluster"].to_list()

    TABLAS = {}
    for c in cats_disp:
        glob = (et_cat.group_by(c).agg(pl.len().alias("n_glob"))
                      .with_columns((pl.col("n_glob") / et_cat.height).alias("sh_glob")))
        porcl = (et_cat.group_by(["cluster", c]).agg(pl.len().alias("n"))
                       .join(et_cat.group_by("cluster").agg(pl.len().alias("n_cl")), on="cluster")
                       .with_columns((pl.col("n") / pl.col("n_cl")).alias("sh_cl"))
                       .join(glob, on=c, how="left")
                       .with_columns((pl.col("sh_cl") / pl.col("sh_glob")).alias("lift"))
                       .sort(["cluster", "lift"], descending=[False, True]))
        TABLAS[c] = porcl
        porcl.write_csv(dir_k / f"composicion_{c}.csv")

        if graficar:
            top = (porcl.filter(pl.col("n") >= PARAM['min_pares_categoria'])
                        .group_by("cluster", maintain_order=True).head(3)
                        .select("cluster", c, "n", "sh_cl", "sh_glob", "lift")
                        .with_columns(pl.col("sh_cl").round(3), pl.col("sh_glob").round(3),
                                      pl.col("lift").round(2)))
            if top.height:
                print(f"\n  {c}: top-3 sobrerrepresentado por cluster "
                      f"(min {PARAM['min_pares_categoria']} pares)")
                print(top)

    concentracion_media = float('nan')
    if "cat3" in TABLAS:
        dom = (TABLAS["cat3"].sort(["cluster", "sh_cl"], descending=[False, True])
                             .group_by("cluster", maintain_order=True).head(1)
                             .join(resumen_cl.select("cluster", "n_pares"), on="cluster"))
        concentracion_media = float((dom["sh_cl"] * dom["n_pares"]).sum() / etiquetas.height)
        print(f"\nconcentracion_cat3_media (ponderada por tamanio de cluster): "
              f"{concentracion_media:.3f}")

    if graficar and "cat3" in TABLAS:
        top_cats = (et_cat.group_by("cat3").agg(pl.len().alias("n"))
                          .sort("n", descending=True).head(14)["cat3"].to_list())
        M = np.ones((len(orden_cl), len(top_cats)))
        mapa = {(r["cluster"], r["cat3"]): r["lift"] for r in TABLAS["cat3"].to_dicts()}
        for i, cl in enumerate(orden_cl):
            for j, ca in enumerate(top_cats):
                M[i, j] = mapa.get((cl, ca), np.nan)
        L = np.log2(np.where(np.isfinite(M) & (M > 0), M, np.nan))
        vmax = float(np.nanmax(np.abs(L))) if np.isfinite(L).any() else 1.0

        fig, ax = plt.subplots(figsize=(1 + .62 * len(top_cats), 1.2 + .46 * len(orden_cl)))
        im = ax.imshow(L, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
        ax.set_xticks(range(len(top_cats)))
        ax.set_xticklabels([str(c)[:14] for c in top_cats], rotation=45, ha="right", fontsize=7.5)
        ax.set_yticks(range(len(orden_cl)))
        ax.set_yticklabels([f"cl {c}" for c in orden_cl], fontsize=8)
        for i in range(len(orden_cl)):
            for j in range(len(top_cats)):
                if np.isfinite(M[i, j]):
                    ax.text(j, i, f"{M[i,j]:.1f}", ha="center", va="center", fontsize=6.5,
                            color=TINTA if abs(L[i, j]) < vmax * .55 else FONDO)
        ax.grid(False)
        ax.set_title(f"k={k} — lift de cat3 por cluster (log2). rojo: sobre, azul: sub",
                     color=TINTA, loc="left", pad=10, fontsize=9)
        fig.colorbar(im, ax=ax, label="log2(lift)", fraction=.025)
        fig.tight_layout()
        guardar(fig, "lift_cat3", subcarpeta=dir_k.name, mostrar=False)

    if graficar:
        N_MUESTRA_PLOT = 60
        ncol = min(3, k)
        nfil = int(np.ceil(k / ncol))
        fig, axes = plt.subplots(nfil, ncol, figsize=(4.6 * ncol, 3.0 * nfil), squeeze=False)
        for pos, cl in enumerate(orden_cl):
            ax = axes[pos // ncol][pos % ncol]
            miembros = np.flatnonzero(LAB == cl)
            cen = RES['centros'][cl]
            T = max(len(cen), int(np.median([len(SERIES[i]) for i in miembros])))
            sub = miembros if len(miembros) <= N_MUESTRA_PLOT else RNG.choice(
                miembros, N_MUESTRA_PLOT, replace=False)
            Mser = np.full((len(sub), T), np.nan)
            for r, i in enumerate(sub):
                v = SERIES[i][:T]
                Mser[r, :len(v)] = v
                ax.plot(np.arange(len(v)), v, color=GRILLA, linewidth=.7, zorder=1)
            with np.errstate(all='ignore'):
                p10 = np.nanpercentile(Mser, 10, axis=0)
                p90 = np.nanpercentile(Mser, 90, axis=0)
            ax.fill_between(np.arange(T), p10, p90, color=SERIE[pos % len(SERIE)],
                            alpha=.18, zorder=2, linewidth=0)
            ax.plot(np.arange(len(cen)), cen, color=SERIE[pos % len(SERIE)], linewidth=2.2, zorder=3)
            fila = resumen_cl.filter(pl.col("cluster") == cl).to_dicts()[0]
            limpiar(ax, f"cl {cl} — {fila['n_pares']:,} ({fila['pct_pares']}%), {fila['pct_tn']}% tn",
                    f"x / {PARAM['escalado']}", "mes desde el inicio del par")
        for pos in range(k, nfil * ncol):
            axes[pos // ncol][pos % ncol].axis("off")
        fig.suptitle(f"k={k} — forma por cluster (banda = percentil 10-90, linea = centroide DBA)",
                     color=TINTA2, fontsize=8.5, y=1.01)
        fig.tight_layout()
        guardar(fig, "forma_por_cluster", subcarpeta=dir_k.name, mostrar=False)

    resumen_cl.write_csv(dir_k / "resumen_clusters.csv")
    fila_comparacion = {
        'k': k, 'k_efectivo': resumen['k_efectivo'],
        'frac_min': round(resumen['frac_min'], 4),
        'silhouette': round(sil, 4), 'n_silhouette': n_sil,
        'inercia_media': round(resumen['inercia_media'], 4),
        'concentracion_cat3_media': round(concentracion_media, 4),
        'iters': len(RES['hist']),
    }
    print(f"[{time.time()-t0:.0f}s]")
    return fila_comparacion, resumen_cl, etiquetas


### Corrida: un ajuste de k-means DTW por cada k de `PARAM['lista_k']`


In [ ]:
resultados_por_k = {}
filas_comparacion = []

for k in PARAM['lista_k']:
    dir_k = DIR_RUN / f"k{k}"
    dir_k.mkdir(parents=True, exist_ok=True)
    fila, resumen_cl, etiquetas = explorar_k(k, dir_k, graficar=PARAM['graficar_por_k'])
    filas_comparacion.append(fila)
    resultados_por_k[k] = {'resumen_cl': resumen_cl, 'etiquetas': etiquetas}
    gc.collect()

comparacion = pl.DataFrame(filas_comparacion)
comparacion.write_csv(DIR_RUN / "comparacion_k.csv")
print(f"\n\nGuardado: {(DIR_RUN / 'comparacion_k.csv').relative_to(BUCKET)}")


### Tabla comparativa y grafico silhouette / balance / concentracion vs k


In [ ]:
with pl.Config(tbl_rows=20, tbl_width_chars=140):
    print(comparacion.sort("k"))

print("\nsilhouette mas alto = clusters mas separados entre si.")
print("frac_min bajo = hay un cluster chiquito (posible outlier aislado, no una forma real).")
print("concentracion_cat3_media alta = el cluster ya se explica por la jerarquia de "
      "categorias existente; baja = agrupa por FORMA cruzando categorias (informacion nueva).")

fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.4), sharex=True)
comp_sorted = comparacion.sort("k")
axes[0].plot(comp_sorted["k"], comp_sorted["silhouette"], color=SERIE[0],
             marker="o", linewidth=1.8)
limpiar(axes[0], "silhouette vs k (mas alto, mejor separados)", "silhouette", "k")
axes[1].plot(comp_sorted["k"], comp_sorted["frac_min"], color=SERIE[1],
             marker="o", linewidth=1.8)
axes[1].axhline(0.02, color=MUDO, linewidth=.8, linestyle=":")
limpiar(axes[1], "cluster mas chico / total (< linea = poco confiable)", "frac_min", "k")
axes[2].plot(comp_sorted["k"], comp_sorted["concentracion_cat3_media"], color=SERIE[2],
             marker="o", linewidth=1.8)
limpiar(axes[2], "concentracion por cat3 dominante (alto = redescubre la jerarquia)",
        "concentracion", "k")
fig.tight_layout()
guardar(fig, "comparacion_k")


### k sugerido y su composicion en detalle

Heuristica simple: el `k` con mayor silhouette entre los que tienen `frac_min >= 0.02` (ningun cluster es un outlier aislado). Es un punto de partida, no una decision automatica -- mira tambien `concentracion_cat3_media` y las figuras de cada `k{K}/` antes de elegir.


In [ ]:
balanceados = comparacion.filter(pl.col("frac_min") >= 0.02)
base = balanceados if balanceados.height else comparacion
K_SUGERIDO = int(base.sort("silhouette", descending=True)["k"][0])

print(f"k sugerido: {K_SUGERIDO}")
print(comparacion.filter(pl.col("k") == K_SUGERIDO))

resumen_cl_sug = resultados_por_k[K_SUGERIDO]['resumen_cl']
print(f"\ncomposicion de cada cluster con k={K_SUGERIDO}:")
print(resumen_cl_sug)
print(f"\nfiguras y csv de esta corrida en: {(DIR_RUN / f'k{K_SUGERIDO}').relative_to(BUCKET)}")


### Guardado: etiquetas del k elegido, listas para unir en 02_FE

Cambia `K_ELEGIDO` a mano si preferis otro k despues de mirar las figuras -- `K_SUGERIDO` es solo el arranque.


In [ ]:
K_ELEGIDO = K_SUGERIDO

etiquetas_elegido = resultados_por_k[K_ELEGIDO]['etiquetas']
NOMBRE = (f"clusters_pc_{PARAM['densificar']}_{PARAM['escalado']}"
          f"_w{PARAM['window']}_k{K_ELEGIDO}_min{PARAM['min_meses']}_corte{PARAM['mes_corte']}")
salida = etiquetas_elegido.select("product_id", "customer_id",
                                  pl.col("cluster").alias(f"cluster_pc_k{K_ELEGIDO}"),
                                  "dist_centroide")
path_out = RUTA_FE / f"{NOMBRE}.parquet"
salida.write_parquet(path_out)

resultado = {
    'notebook': 'dtw_nuevo_explorar_k',
    'idea': 'compara varios k de k-means DTW a nivel par cliente-producto sobre las '
            'mismas series (armadas una sola vez), con composicion por categoria y lift',
    'param': PARAM,
    'k_sugerido': K_SUGERIDO,
    'k_elegido': K_ELEGIDO,
    'comparacion_k': comparacion.to_dicts(),
    'n_pares_clusterizados': int(etiquetas_elegido.height),
    'n_pares_totales': int(vida.height),
    'archivo_salida': str(path_out),
    'semilla': PARAM['semilla'],
}
with open(DIR_RUN / "resultado_comparacion_k.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)

print(f"Etiquetas (k={K_ELEGIDO}): {path_out}")
print(f"  {salida.height:,} pares de {vida.height:,} "
      f"({100*salida.height/vida.height:.0f}%); el resto queda nulo al joinear")
print(f"\nPara usarlo en 02_FE:")
print(f"  cl = pl.read_parquet(r'{path_out}')")
print(f"  df = df.join(cl.drop('dist_centroide'), on=['product_id','customer_id'], how='left')")
